# COCO 200 Dataset Downloader
## এই notebook টা Google Colab-এ run করো

**কী করবে:**
1. COCO 2017 val annotation থেকে person-সহ ইমেজগুলো খুঁজে বের করবে
2. সেরা ২০০টি ইমেজ download করবে (variety নিশ্চিত করতে person count মিক্স করা)
3. `coco_200_persons.zip` বানাবে
4. তোমার Google Drive-এ save করবে

**তারপর:** সেই zip ফাইলটা Kaggle-এ Dataset হিসেবে upload করো।

In [ ]:
# ── Step 0: Google Drive mount করো (ফাইল সেভ করার জন্য) ──────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
# ── Step 1: Dependencies ────────────────────────────────────────────────────
!pip install pycocotools -q
import os, json, zipfile, shutil, random, urllib.request, time
from pathlib import Path
from pycocotools.coco import COCO
print('Done!')

In [ ]:
# ── Step 2: COCO val2017 annotation download ────────────────────────────────
WORK_DIR   = '/content/coco_download'
OUTPUT_DIR = '/content/coco_200_persons'
os.makedirs(WORK_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

ANN_ZIP = f'{WORK_DIR}/annotations.zip'
ANN_URL = 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'

print('Downloading COCO 2017 annotations (~241 MB)...')
print('This takes ~3-5 minutes on Colab...')

if not os.path.exists(f'{WORK_DIR}/annotations/instances_val2017.json'):
    urllib.request.urlretrieve(ANN_URL, ANN_ZIP)
    print('Extracting...')
    import zipfile
    with zipfile.ZipFile(ANN_ZIP, 'r') as z:
        z.extractall(WORK_DIR)
    print('Annotation extracted!')
else:
    print('Annotation already exists — skipping download')

ANN_FILE = f'{WORK_DIR}/annotations/instances_val2017.json'
coco = COCO(ANN_FILE)
print('COCO API loaded!')

In [ ]:
# ── Step 3: Person-সহ ইমেজ select করো (variety সহ) ────────────────────────
TOTAL_IMAGES  = 200
RANDOM_SEED   = 42
random.seed(RANDOM_SEED)

# Category 1 = person
person_cat_id = coco.getCatIds(catNms=['person'])[0]
print(f'Person category ID: {person_cat_id}')

# সব person annotation-এর image_id গুলো নাও
ann_ids   = coco.getAnnIds(catIds=[person_cat_id], iscrowd=False)
anns      = coco.loadAnns(ann_ids)

# প্রতিটা image-এ কতজন person আছে গোনো
from collections import Counter
img_person_count = Counter(a['image_id'] for a in anns)

# Minimum area threshold — খুব ছোট person (crowd দূরে থাকুক)
MIN_PERSON_AREA = 5000  # pixels

# Image per person-count bucket (variety নিশ্চিত করতে)
buckets = {
    1:  [],   # 1 person
    2:  [],   # 2 persons
    3:  [],   # 3 persons
    '4+': [], # 4+ persons
}

for img_id, count in img_person_count.items():
    # Check that at least one person annotation has large enough area
    img_anns = [a for a in anns if a['image_id'] == img_id and a['area'] >= MIN_PERSON_AREA]
    if len(img_anns) == 0:
        continue
    if   count == 1:  buckets[1].append(img_id)
    elif count == 2:  buckets[2].append(img_id)
    elif count == 3:  buckets[3].append(img_id)
    else:             buckets['4+'].append(img_id)

print(f'\nBucket sizes:')
for k, v in buckets.items():
    print(f'  {k} person(s): {len(v)} images available')

# 200 images proportionally নাও
counts_target = {1: 60, 2: 60, 3: 40, '4+': 40}
selected_ids = []
for key, target in counts_target.items():
    pool   = buckets[key]
    random.shuffle(pool)
    take   = min(target, len(pool))
    selected_ids.extend(pool[:take])
    print(f'  Selected {take}/{target} from bucket {key}')

# ঘাটতি থাকলে যেকোনো bucket থেকে নাও
if len(selected_ids) < TOTAL_IMAGES:
    all_ids = list(img_person_count.keys())
    remaining = [x for x in all_ids if x not in set(selected_ids)]
    random.shuffle(remaining)
    need = TOTAL_IMAGES - len(selected_ids)
    selected_ids.extend(remaining[:need])
    print(f'  Filled remaining {need} from pool')

selected_ids = selected_ids[:TOTAL_IMAGES]
print(f'\nTotal selected: {len(selected_ids)} images')

In [ ]:
# ── Step 4: Images download করো ─────────────────────────────────────────────
COCO_IMG_URL = 'http://images.cocodataset.org/val2017/'
img_infos    = coco.loadImgs(selected_ids)

failed = []
for i, info in enumerate(img_infos, 1):
    fname  = info['file_name']
    out_p  = f'{OUTPUT_DIR}/{fname}'
    url    = COCO_IMG_URL + fname

    if os.path.exists(out_p):
        print(f'[{i:>3}/200] EXISTS: {fname}')
        continue
    try:
        urllib.request.urlretrieve(url, out_p)
        if i % 20 == 0 or i <= 5:
            size_kb = os.path.getsize(out_p) // 1024
            print(f'[{i:>3}/200] ✓ {fname}  ({size_kb} KB)')
    except Exception as e:
        print(f'[{i:>3}/200] ✗ FAILED: {fname} — {e}')
        failed.append(fname)
    time.sleep(0.05)  # polite rate limit

downloaded = len(list(Path(OUTPUT_DIR).glob('*.jpg')))
print(f'\n✅ Downloaded: {downloaded} images')
if failed:
    print(f'Failed ({len(failed)}): {failed[:5]}')

In [ ]:
# ── Step 5: Metadata JSON save করো (image_id, person count info) ────────────
metadata = []
for info in img_infos:
    img_id  = info['id']
    n_large = len([a for a in anns
                   if a['image_id'] == img_id and a['area'] >= MIN_PERSON_AREA])
    metadata.append({
        'file_name':    info['file_name'],
        'image_id':     img_id,
        'width':        info['width'],
        'height':       info['height'],
        'n_persons_gt': img_person_count[img_id],   # ground truth count
        'n_large_gt':   n_large,
    })

meta_path = f'{OUTPUT_DIR}/metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved: {meta_path}')
print(f'Sample entry: {metadata[0]}')

In [ ]:
# ── Step 6: ZIP বানাও + Google Drive-এ save করো ─────────────────────────────
ZIP_NAME   = 'coco_200_persons.zip'
DRIVE_PATH = f'/content/drive/MyDrive/{ZIP_NAME}'
LOCAL_ZIP  = f'/content/{ZIP_NAME}'

print(f'Creating zip: {ZIP_NAME} ...')
with zipfile.ZipFile(LOCAL_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in Path(OUTPUT_DIR).iterdir():
        zf.write(fpath, fpath.name)

size_mb = os.path.getsize(LOCAL_ZIP) / 1e6
print(f'ZIP created: {size_mb:.1f} MB')

print(f'Copying to Google Drive...')
shutil.copy(LOCAL_ZIP, DRIVE_PATH)
print(f'\n✅ Done! File saved to:')
print(f'   Google Drive → {ZIP_NAME}')
print()
print('── NEXT STEPS ───────────────────────────────────────────────')
print('1. Google Drive খুলো')
print('2. coco_200_persons.zip ফাইলটা download করো')
print('3. Kaggle → Datasets → New Dataset → Upload zip')
print('4. Dataset name: coco-200-persons')
print('5. তারপর Research notebook-এ এই path ব্যবহার করো:')
print('   /kaggle/input/coco-200-persons/')
print('─────────────────────────────────────────────────────────────')

In [ ]:
# ── Step 7: Quick preview — কয়েকটা ইমেজ দেখো ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter

# Person count distribution
dist = Counter(m['n_large_gt'] for m in metadata)
print('Person count distribution in selected 200 images:')
for k in sorted(dist):
    bar = '█' * dist[k]
    print(f'  {k} person(s): {dist[k]:>3} images  {bar}')

# Show 8 sample images
sample_files = random.sample(list(Path(OUTPUT_DIR).glob('*.jpg')), 8)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle('Sample Images from Dataset (200 COCO val2017 with persons)',
             fontsize=13, fontweight='bold')
for ax, fp in zip(axes.flatten(), sample_files):
    meta = next(m for m in metadata if m['file_name'] == fp.name)
    ax.imshow(mpimg.imread(str(fp)))
    ax.set_title(f"{fp.name}\n{meta['n_large_gt']} person(s)", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'/content/drive/MyDrive/dataset_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('Preview saved to Drive!')